# Co2L: Evaluación Class-IL (NCM)


In [1]:
import torch
import os
from torch.utils.data import ConcatDataset, DataLoader
from models import CNN, Co2LModel
from dataloaders import SequentialCIFAR10
from utils import load_co2l_model
from utils_class_il import compute_class_prototypes, evaluate_ncm

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"Usando dispositivo: {device}")

Usando dispositivo: mps


In [2]:
seq_cifar = SequentialCIFAR10(batch_size=128)
NUM_TASKS = 5

for task_id in range(NUM_TASKS):
    ckpt_path = f"checkpoints/co2l/task_{task_id}.pth"
    if not os.path.exists(ckpt_path):
        continue
    
    print(f"\n--- Evaluando Estado tras Tarea {task_id} ---")
    
    # 1. Cargar el modelo de la tarea actual
    model = load_co2l_model(ckpt_path, device)
    
    # 2. RECALCULAR TODOS los prototipos (0 hasta task_id) con el modelo actual
    # Para un estudio técnico, usamos los datasets completos para ver el potencial del backbone
    all_train_datasets = []
    for tid in range(task_id + 1):
        all_train_datasets.append(seq_cifar.get_task_train_dataset(tid, remap_labels=False))
    
    combined_train_loader = DataLoader(ConcatDataset(all_train_datasets), batch_size=128, shuffle=False)
    current_prototypes = compute_class_prototypes(model, combined_train_loader, device)
    
    # 3. Evaluar en TEST combinado
    all_test_datasets = []
    for tid in range(task_id + 1):
        all_test_datasets.append(seq_cifar.get_task_test_dataset(tid))
    
    combined_test_loader = DataLoader(ConcatDataset(all_test_datasets), batch_size=128, shuffle=False)
    
    acc_class_il = evaluate_ncm(model, combined_test_loader, current_prototypes, device)
    print(f"Precisión Class-IL (Recalculada): {acc_class_il:.2f}%")


--- Evaluando Estado tras Tarea 0 ---
Precisión Class-IL (Recalculada): 95.75%

--- Evaluando Estado tras Tarea 1 ---
Precisión Class-IL (Recalculada): 71.08%

--- Evaluando Estado tras Tarea 2 ---
Precisión Class-IL (Recalculada): 58.35%

--- Evaluando Estado tras Tarea 3 ---
Precisión Class-IL (Recalculada): 54.62%

--- Evaluando Estado tras Tarea 4 ---
Precisión Class-IL (Recalculada): 50.82%
